In [ ]:
# -*- coding: utf-8 -*-
"""
Development of An Affordable Non-Invasive Anemia Screening Device
Using Multi-Wavelength Optical Sensing, Edge Machine Learning
and Adaptive Skin-Tone Adaptation

Updated pipeline:
  - w730 → w530  (530 nm replaces 730 nm on physical device)
  - Skin-tone classification from 530 nm DC ratio (melanin index)
  - Fitzpatrick-group-specific Hb correction applied after RF prediction
  - Skin-tone group label stored as feature and used in per-group evaluation
"""

# =============================================================================
# CELL 2 — Extract & locate dataset
# =============================================================================

import os, zipfile, glob

ZIP_PATH    = "/content/Hb_PPG_Dataset.zip"
EXTRACT_DIR = "/content/Hb_PPG_Dataset"
OUTPUT_DIR  = "/content/outputs"
CACHE_DIR   = "/content/cache"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

if not os.path.isdir(EXTRACT_DIR):
    if os.path.isfile(ZIP_PATH):
        print(f"Extracting {ZIP_PATH} ...")
        with zipfile.ZipFile(ZIP_PATH, "r") as z:
            z.extractall(EXTRACT_DIR)
        print("Extraction complete.")
    else:
        raise FileNotFoundError(f"\n⚠ Zip not found at: {ZIP_PATH}")
else:
    print(f"Dataset folder already exists: {EXTRACT_DIR}")

candidates = glob.glob(os.path.join(EXTRACT_DIR, "**", "data_csv"), recursive=True)
if candidates:
    DATA_DIR = candidates[0]
else:
    for root, dirs, files in os.walk(EXTRACT_DIR):
        csv_files = [f for f in files if f.endswith(".csv") and f.replace(".csv","").isdigit()]
        if csv_files:
            DATA_DIR = root
            break
    else:
        raise FileNotFoundError(f"\n⚠ Could not find data_csv inside {EXTRACT_DIR}")

print(f"DATA_DIR : {DATA_DIR}")
print(f"CSV files: {len([f for f in os.listdir(DATA_DIR) if f.endswith('.csv')])}")


# =============================================================================
# CELL 3 — Imports
# =============================================================================

import json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.signal import butter, filtfilt, iirnotch, find_peaks, welch
from scipy.stats import skew, kurtosis, pearsonr, entropy as scipy_entropy
from scipy.integrate import trapezoid
from typing import Optional
import warnings
warnings.filterwarnings("ignore")

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import (KFold, cross_val_predict, GridSearchCV,
                                      LeaveOneOut, cross_validate,
                                      RandomizedSearchCV)
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.feature_selection import RFECV, SelectKBest, f_regression
from sklearn.isotonic import IsotonicRegression
from scipy.stats import randint, uniform
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor


# =============================================================================
# CELL 4 — Hb Labels & Configuration
# =============================================================================

SUBJECT_HB = {
    "1":   16.4, "2":   11.1, "3":   12.4, "4":   15.7, "5":   13.7,
    "6":   11.4, "7":   15.2, "8":   15.8, "9":   17.0, "10":  13.1,
    "11":  12.8, "12":  13.5, "13":  13.4, "14":  15.6, "15":  16.1,
    "16":  14.6, "17":  13.2, "18":  13.0, "19":  12.2, "20":  12.7,
    "21":  14.8, "22":  11.7, "23":  12.2, "24":  14.7, "25":  15.3,
    "26":  12.1, "27":  14.4, "28":  14.9, "29":  15.5, "30":  13.5,
    "31":  15.6, "32":  16.0, "33":  13.0, "34":  13.6, "35":  12.0,
    "36":  15.9, "37":  14.8, "38":  15.7, "39":  15.0, "40":  16.1,
    "41":  12.9, "42":  15.2, "43":  13.1, "44":  15.1, "45":  14.5,
    "46":  15.7, "47":  13.3, "48":  12.8, "49":  13.0, "50":  15.4,
    "51":  13.2, "52":  16.9, "53":  12.1, "54":  13.2, "55":  15.9,
    "56":  15.3, "57":  14.4, "58":  15.5, "59":  12.1, "60":  16.2,
    "61":  13.8, "62":  15.7, "63":  12.7, "64":  12.5, "65":  15.7,
    "66":  14.3, "67":  13.8, "68":  13.9, "69":  13.9, "70":  14.0,
    "71":  13.1, "72":  13.0, "73":  12.0, "74":  12.5, "75":  13.4,
    "76":  13.5, "77":  13.0, "78":  14.0, "79":  12.7, "80":  12.9,
    "81":  12.8, "82":  12.2, "83":  15.2, "84":  12.2, "85":  15.1,
    "86":  13.1, "87":  13.0, "88":  13.5, "89":  16.0, "90":  13.4,
    "91":  15.3, "92":  13.6, "93":  12.8, "94":  16.1, "95":  14.1,
    "96":  14.5, "97":  16.2, "98":  13.0, "99":  13.2,
    "101": 15.0, "102": 13.5, "103": 12.5, "104": 13.6, "105": 15.3,
    "106": 13.4, "107": 13.3, "108": 13.8, "109": 15.0, "110": 13.5,
    "111": 12.2, "112": 12.9, "113": 15.6, "114": 15.9, "115": 15.3,
    "116": 14.5, "117": 13.1, "118": 12.6, "119": 13.4, "120": 14.7,
    "121": 14.2, "122": 13.3, "123": 13.8, "124": 13.1, "125": 14.1,
    "126": 11.2, "127": 13.8, "128": 15.1, "129": 13.8, "130": 14.3,
    "131": 16.8, "133": 12.2, "134": 14.3, "135": 13.7, "136": 13.8,
    "137": 15.5, "138": 13.7, "139": 12.8, "140": 12.3, "141": 15.6,
    "142": 13.7, "143": 12.0, "144": 14.6, "145": 14.2, "146": 12.4,
    "147": 15.0, "148": 13.7, "149": 13.5, "150": 12.4, "151": 13.6,
    "152": 11.0, "153": 17.2, "154": 12.0, "155": 14.0, "156": 14.5,
    "157": 13.5, "158": 11.9, "159": 11.6, "160": 13.5, "161": 14.1,
    "162": 13.1, "163": 13.3, "164": 15.5, "165": 14.8, "166": 13.6,
    "167": 12.9, "168": 13.6, "169": 13.8, "170": 15.0, "171": 14.8,
    "172": 15.5, "173": 15.6, "174": 13.4, "175": 13.3, "176": 15.6,
    "177": 14.2, "178": 14.0, "179": 13.6, "180": 13.8, "181": 15.1,
    "182": 15.5, "183": 14.8, "184": 15.5, "185": 15.1, "186": 15.3,
    "187": 14.3, "188": 15.5, "189":  9.6, "190": 12.4, "191": 12.3,
    "192": 12.8, "193": 11.9, "194": 17.3, "195": 13.7, "196": 16.4,
    "197": 14.6, "198": 16.2, "199": 13.1, "200": 13.6, "201": 15.7,
    "202": 14.3, "203": 15.0, "204": 14.0, "205": 12.6, "206": 15.1,
    "207": 13.4, "208": 12.5, "209": 16.3, "210": 13.5, "211": 15.9,
    "212": 13.5, "213": 13.0, "214": 13.7, "215": 12.9, "216": 15.9,
    "217": 12.3, "218": 13.9, "219": 13.7, "220": 15.8, "221":  8.5,
    "222": 13.7, "223": 16.3, "224": 13.5, "225": 14.1, "226": 13.8,
    "227": 14.0, "228":  8.7, "229": 12.2, "230": 12.7, "231": 16.4,
    "232": 14.7, "233": 10.3, "234": 16.9, "235": 10.8, "236": 13.6,
    "237": 13.0, "238": 13.3, "239": 12.9, "240": 15.2, "241": 13.3,
    "242": 16.0, "243": 14.9, "244": 14.1, "245": 12.2, "246": 14.7,
    "247": 12.6, "248": 13.4, "249": 13.5, "250": 13.2, "251": 11.8,
    "252": 14.0,
}

# Fitzpatrick skin-tone labels (None = inferred from 530nm melanin index)
# If you have ground-truth labels, fill them in here as integers 1-6.
SUBJECT_FITZPATRICK = {sid: None for sid in SUBJECT_HB}

FS          = 100
WINDOW_SAMP = 800     # 8 s × 100 Hz
STEP_SAMP   = 200     # 75% overlap

# ── UPDATED: w530 replaces w730 on the physical device ───────────────────────
# The dataset column "w730" is read but relabelled internally as "w530"
# because the physical LED is 530 nm (green/yellow — peak melanin absorption).
# All feature names using "530" refer to the green channel on the device.
WL_LABELS       = ["w530", "w660", "w850", "w940"]   # w530 = physical 530nm LED
WL_LABELS_CSV   = ["w730", "w660", "w850", "w940"]   # CSV column name in dataset
WL_CSV_TO_LABEL = {"w730": "w530", "w660": "w660",
                   "w850": "w850", "w940": "w940"}    # remapping at load time

EXCLUDE              = {"100", "132"}
WHO_THRESHOLD_GDPL   = 1.0

# =============================================================================
# SKIN-TONE ADAPTATION CONFIGURATION
# =============================================================================
# Physical basis:
#   530 nm (green) is near the peak melanin absorption (~500-570 nm).
#   Higher melanin → lower DC intensity at 530 nm → higher MI_530_940 ratio.
#   We use the DC ratio DC_530 / DC_940 (Melanin Index proxy) to classify
#   skin tone into 3 groups (light / medium / dark) without needing a camera.
#
# Fitzpatrick mapping:
#   Group 1 (light)  → Fitzpatrick I-II   MI_530_940 < SKIN_THR_LOW
#   Group 2 (medium) → Fitzpatrick III-IV  SKIN_THR_LOW ≤ MI < SKIN_THR_HIGH
#   Group 3 (dark)   → Fitzpatrick V-VI   MI_530_940 ≥ SKIN_THR_HIGH
#
# Thresholds are data-driven (set after running compute_melanin_thresholds()).
# These defaults work for the Hb_PPG_Dataset; tune on your own cohort.
SKIN_THR_LOW  = 0.72   # MI below this → light skin
SKIN_THR_HIGH = 0.88   # MI above this → dark skin

# Per-group linear Hb correction offsets (fitted by fit_skin_tone_correction()).
# Format: {group_int: (slope, intercept)}
# Initialised to identity; updated by fit_skin_tone_correction().
SKIN_CORRECTION = {
    1: (1.0, 0.0),   # light
    2: (1.0, 0.0),   # medium
    3: (1.0, 0.0),   # dark
}

GROUP_NAMES = {1: "Light (I-II)", 2: "Medium (III-IV)", 3: "Dark (V-VI)"}


# =============================================================================
# CELL 4b — Demographic Data
# =============================================================================

SUBJECT_DEMOGRAPHICS = {
    "1": {"age":24,"sex":1}, "2": {"age":23,"sex":0}, "3": {"age":23,"sex":1},
    "4": {"age":23,"sex":1}, "5": {"age":26,"sex":1}, "6": {"age":25,"sex":0},
    "7": {"age":24,"sex":1}, "8": {"age":24,"sex":1}, "9": {"age":23,"sex":1},
    "10": {"age":23,"sex":0}, "11": {"age":26,"sex":0}, "12": {"age":25,"sex":0},
    "13": {"age":23,"sex":0}, "14": {"age":25,"sex":1}, "15": {"age":24,"sex":1},
    "16": {"age":23,"sex":1}, "17": {"age":23,"sex":0}, "18": {"age":27,"sex":0},
    "19": {"age":24,"sex":0}, "20": {"age":26,"sex":0}, "21": {"age":25,"sex":1},
    "22": {"age":23,"sex":0}, "23": {"age":23,"sex":0}, "24": {"age":25,"sex":1},
    "25": {"age":25,"sex":1}, "26": {"age":23,"sex":0}, "27": {"age":24,"sex":1},
    "28": {"age":25,"sex":1}, "29": {"age":23,"sex":1}, "30": {"age":24,"sex":1},
    "31": {"age":23,"sex":1}, "32": {"age":24,"sex":1}, "33": {"age":24,"sex":0},
    "34": {"age":24,"sex":1}, "35": {"age":24,"sex":0}, "36": {"age":24,"sex":1},
    "37": {"age":24,"sex":1}, "38": {"age":23,"sex":1}, "39": {"age":25,"sex":1},
    "40": {"age":24,"sex":1}, "41": {"age":23,"sex":0}, "42": {"age":23,"sex":1},
    "43": {"age":24,"sex":1}, "44": {"age":23,"sex":1}, "45": {"age":22,"sex":1},
    "46": {"age":23,"sex":1}, "47": {"age":23,"sex":0}, "48": {"age":24,"sex":0},
    "49": {"age":24,"sex":0}, "50": {"age":24,"sex":1}, "51": {"age":21,"sex":0},
    "52": {"age":24,"sex":1}, "53": {"age":25,"sex":0}, "54": {"age":24,"sex":0},
    "55": {"age":24,"sex":1}, "56": {"age":24,"sex":1}, "57": {"age":24,"sex":0},
    "58": {"age":24,"sex":1}, "59": {"age":24,"sex":0}, "60": {"age":23,"sex":1},
    "61": {"age":25,"sex":0}, "62": {"age":25,"sex":1}, "63": {"age":24,"sex":0},
    "64": {"age":24,"sex":0}, "65": {"age":23,"sex":1}, "66": {"age":52,"sex":0},
    "67": {"age":23,"sex":0}, "68": {"age":26,"sex":0}, "69": {"age":24,"sex":0},
    "70": {"age":23,"sex":0}, "71": {"age":23,"sex":0}, "72": {"age":24,"sex":0},
    "73": {"age":24,"sex":0}, "74": {"age":24,"sex":0}, "75": {"age":23,"sex":0},
    "76": {"age":24,"sex":0}, "77": {"age":23,"sex":0}, "78": {"age":85,"sex":0},
    "79": {"age":25,"sex":0}, "80": {"age":62,"sex":0}, "81": {"age":25,"sex":0},
    "82": {"age":53,"sex":0}, "83": {"age":25,"sex":1}, "84": {"age":24,"sex":0},
    "85": {"age":71,"sex":1}, "86": {"age":86,"sex":1}, "87": {"age":82,"sex":1},
    "88": {"age":74,"sex":0}, "89": {"age":44,"sex":1}, "90": {"age":74,"sex":0},
    "91": {"age":56,"sex":1}, "92": {"age":66,"sex":0}, "93": {"age":78,"sex":0},
    "94": {"age":80,"sex":1}, "95": {"age":73,"sex":0}, "96": {"age":70,"sex":0},
    "97": {"age":60,"sex":1}, "98": {"age":87,"sex":0}, "99": {"age":81,"sex":0},
    "100": {"age":73,"sex":0}, "101": {"age":69,"sex":1}, "102": {"age":80,"sex":0},
    "103": {"age":64,"sex":0}, "104": {"age":78,"sex":0}, "105": {"age":85,"sex":1},
    "106": {"age":79,"sex":0}, "107": {"age":64,"sex":0}, "108": {"age":78,"sex":0},
    "109": {"age":49,"sex":1}, "110": {"age":58,"sex":0}, "111": {"age":84,"sex":0},
    "112": {"age":88,"sex":1}, "113": {"age":73,"sex":1}, "114": {"age":51,"sex":1},
    "115": {"age":85,"sex":1}, "116": {"age":68,"sex":0}, "117": {"age":86,"sex":0},
    "118": {"age":67,"sex":1}, "119": {"age":61,"sex":0}, "120": {"age":68,"sex":1},
    "121": {"age":51,"sex":1}, "122": {"age":64,"sex":0}, "123": {"age":74,"sex":0},
    "124": {"age":52,"sex":0}, "125": {"age":57,"sex":0}, "126": {"age":88,"sex":1},
    "127": {"age":68,"sex":1}, "128": {"age":69,"sex":1}, "129": {"age":59,"sex":0},
    "130": {"age":65,"sex":1}, "131": {"age":63,"sex":1}, "132": {"age":71,"sex":0},
    "133": {"age":65,"sex":0}, "134": {"age":67,"sex":1}, "135": {"age":67,"sex":0},
    "136": {"age":62,"sex":0}, "137": {"age":67,"sex":1}, "138": {"age":66,"sex":0},
    "139": {"age":68,"sex":0}, "140": {"age":61,"sex":0}, "141": {"age":65,"sex":1},
    "142": {"age":65,"sex":0}, "143": {"age":60,"sex":0}, "144": {"age":69,"sex":1},
    "145": {"age":61,"sex":1}, "146": {"age":61,"sex":0}, "147": {"age":63,"sex":1},
    "148": {"age":67,"sex":0}, "149": {"age":67,"sex":0}, "150": {"age":61,"sex":0},
    "151": {"age":50,"sex":0}, "152": {"age":78,"sex":1}, "153": {"age":24,"sex":1},
    "154": {"age":66,"sex":0}, "155": {"age":75,"sex":0}, "156": {"age":75,"sex":0},
    "157": {"age":83,"sex":1}, "158": {"age":75,"sex":1}, "159": {"age":78,"sex":0},
    "160": {"age":80,"sex":0}, "161": {"age":76,"sex":1}, "162": {"age":50,"sex":0},
    "163": {"age":50,"sex":0}, "164": {"age":76,"sex":1}, "165": {"age":61,"sex":0},
    "166": {"age":64,"sex":0}, "167": {"age":67,"sex":0}, "168": {"age":73,"sex":1},
    "169": {"age":75,"sex":0}, "170": {"age":86,"sex":1}, "171": {"age":82,"sex":0},
    "172": {"age":61,"sex":1}, "173": {"age":62,"sex":0}, "174": {"age":90,"sex":1},
    "175": {"age":68,"sex":1}, "176": {"age":40,"sex":1}, "177": {"age":56,"sex":0},
    "178": {"age":64,"sex":0}, "179": {"age":59,"sex":0}, "180": {"age":69,"sex":1},
    "181": {"age":61,"sex":0}, "182": {"age":70,"sex":1}, "183": {"age":66,"sex":0},
    "184": {"age":49,"sex":1}, "185": {"age":32,"sex":1}, "186": {"age":44,"sex":1},
    "187": {"age":51,"sex":1}, "188": {"age":49,"sex":1}, "189": {"age":41,"sex":0},
    "190": {"age":42,"sex":0}, "191": {"age":36,"sex":0}, "192": {"age":45,"sex":0},
    "193": {"age":46,"sex":0}, "194": {"age":57,"sex":1}, "195": {"age":45,"sex":0},
    "196": {"age":28,"sex":1}, "197": {"age":51,"sex":1}, "198": {"age":32,"sex":1},
    "199": {"age":44,"sex":0}, "200": {"age":55,"sex":0}, "201": {"age":37,"sex":1},
    "202": {"age":39,"sex":0}, "203": {"age":40,"sex":1}, "204": {"age":52,"sex":0},
    "205": {"age":30,"sex":0}, "206": {"age":27,"sex":1}, "207": {"age":41,"sex":0},
    "208": {"age":42,"sex":0}, "209": {"age":34,"sex":1}, "210": {"age":30,"sex":0},
    "211": {"age":28,"sex":1}, "212": {"age":51,"sex":1}, "213": {"age":27,"sex":0},
    "214": {"age":54,"sex":0}, "215": {"age":46,"sex":0}, "216": {"age":54,"sex":1},
    "217": {"age":41,"sex":0}, "218": {"age":30,"sex":0}, "219": {"age":46,"sex":1},
    "220": {"age":51,"sex":1}, "221": {"age":29,"sex":0}, "222": {"age":30,"sex":1},
    "223": {"age":47,"sex":1}, "224": {"age":44,"sex":0}, "225": {"age":52,"sex":1},
    "226": {"age":55,"sex":0}, "227": {"age":51,"sex":0}, "228": {"age":37,"sex":0},
    "229": {"age":31,"sex":0}, "230": {"age":51,"sex":0}, "231": {"age":53,"sex":1},
    "232": {"age":53,"sex":1}, "233": {"age":55,"sex":0}, "234": {"age":48,"sex":1},
    "235": {"age":48,"sex":0}, "236": {"age":40,"sex":0}, "237": {"age":44,"sex":0},
    "238": {"age":79,"sex":0}, "239": {"age":50,"sex":0}, "240": {"age":42,"sex":1},
    "241": {"age":39,"sex":0}, "242": {"age":45,"sex":1}, "243": {"age":53,"sex":1},
    "244": {"age":37,"sex":0}, "245": {"age":49,"sex":0}, "246": {"age":44,"sex":1},
    "247": {"age":45,"sex":0}, "248": {"age":43,"sex":0}, "249": {"age":37,"sex":0},
    "250": {"age":43,"sex":0}, "251": {"age":38,"sex":0}, "252": {"age":33,"sex":0},
}


def get_demo_features(sid: str) -> dict:
    d   = SUBJECT_DEMOGRAPHICS.get(sid, {})
    def f(v): return float(v) if v is not None else np.nan
    return {
        "demo_age": f(d.get("age")),
        "demo_sex": f(d.get("sex")),
    }


# =============================================================================
# WAVELENGTH MISMATCH CORRECTION
# Updated: w530 on device vs w730 in dataset (200 nm shift!)
# The correction factor accounts for the different absorption coefficient.
# For Beer-Lambert: A = ε × c × l  → correction scales for ε difference.
# Melanin extinction: ε_530 ≈ 3× ε_730 → DC_530 ≈ DC_730 / 3 at same melanin.
# This is handled in features by using ratios, so raw correction = 1.0.
# The spectral slope features are updated to use [530, 660, 850, 940] nm.
# =============================================================================

DEVICE_WL  = {"w530": 530, "w660": 660, "w850": 850, "w940": 940}
DATASET_WL = {"w530": 730, "w660": 660, "w850": 850, "w940": 940}
# NOTE: dataset has 730nm data mapped to w530 label.
# The 200nm mismatch is intentional: we use the ratio features which
# are self-normalising. The WL_CORRECTION is set to 1.0 for all channels
# since we treat 530nm as a melanin-specific channel separately.
DEPSION_DWL = {"w530": 0.0, "w660": 0.080, "w850": 0.012, "w940": 0.008}

def get_wavelength_correction(lbl: str) -> float:
    delta = DEVICE_WL[lbl] - DATASET_WL[lbl]
    if delta == 0:
        return 1.0
    correction = 1.0 + delta * DEPSION_DWL[lbl]
    return max(0.5, min(2.0, correction))

WL_CORRECTION = {lbl: get_wavelength_correction(lbl) for lbl in WL_LABELS}
print("\n── Wavelength Correction Factors ────────────────────────────────")
for lbl, factor in WL_CORRECTION.items():
    dev_nm  = DEVICE_WL[lbl]
    data_nm = DATASET_WL[lbl]
    print(f"  {lbl}: device={dev_nm}nm, dataset={data_nm}nm, "
          f"correction={factor:.4f}")


# =============================================================================
# CELL 4c — Skin-Tone Classification from 530 nm
# =============================================================================

def compute_melanin_index(dc_530: np.ndarray, dc_940: np.ndarray) -> float:
    """
    Melanin Index (MI) = DC_530 / DC_940.

    Physical basis:
      530 nm is near peak melanin absorption (~500-570 nm).
      940 nm has minimal melanin absorption.
      Higher MI → more melanin → darker skin tone.

    Range in practice: ~0.4 (very light) to ~1.2 (very dark).
    """
    return float(np.mean(np.abs(dc_530)) / (np.mean(np.abs(dc_940)) + 1e-10))


def classify_skin_tone(mi: float) -> int:
    """
    Classify Fitzpatrick group from Melanin Index.
      1 = Light   (Fitzpatrick I-II)   MI < SKIN_THR_LOW
      2 = Medium  (Fitzpatrick III-IV) SKIN_THR_LOW ≤ MI < SKIN_THR_HIGH
      3 = Dark    (Fitzpatrick V-VI)   MI ≥ SKIN_THR_HIGH
    """
    if mi < SKIN_THR_LOW:
        return 1
    elif mi < SKIN_THR_HIGH:
        return 2
    else:
        return 3


def compute_melanin_thresholds(feat_df: pd.DataFrame) -> tuple:
    """
    Data-driven threshold selection using k-means (k=3) on the MI distribution.
    Prints the computed thresholds so you can update SKIN_THR_LOW/HIGH.
    """
    from sklearn.cluster import KMeans

    mi_vals = feat_df["mi_530_940"].values.reshape(-1, 1)
    km = KMeans(n_clusters=3, random_state=42, n_init=10)
    km.fit(mi_vals)
    centres = sorted(km.cluster_centers_.flatten())
    thr_low  = (centres[0] + centres[1]) / 2
    thr_high = (centres[1] + centres[2]) / 2
    print(f"\n── Auto skin-tone thresholds ─────────────────────────────────")
    print(f"  K-means centres: {[f'{c:.3f}' for c in centres]}")
    print(f"  SKIN_THR_LOW  = {thr_low:.3f}  (update in config)")
    print(f"  SKIN_THR_HIGH = {thr_high:.3f}  (update in config)")
    return thr_low, thr_high


def fit_skin_tone_correction(feat_df: pd.DataFrame,
                              y_pred_col: str = "hb_pred_raw") -> dict:
    """
    Fit a per-group linear correction:  Hb_corrected = slope × Hb_raw + intercept
    Minimises residual bias within each skin-tone group.

    Uses ordinary least squares (scipy.stats.linregress) per group.
    Returns the updated SKIN_CORRECTION dict.
    """
    from scipy.stats import linregress

    correction = {}
    print("\n── Per-group linear Hb correction ───────────────────────────")
    for g in [1, 2, 3]:
        mask = feat_df["skin_group"] == g
        n    = mask.sum()
        if n < 5:
            print(f"  Group {g} ({GROUP_NAMES[g]}): only {n} subjects — using identity")
            correction[g] = (1.0, 0.0)
            continue
        y_true = feat_df.loc[mask, "hb_true"].values
        y_raw  = feat_df.loc[mask, y_pred_col].values
        slope, intercept, r, p, se = linregress(y_raw, y_true)
        correction[g] = (slope, intercept)
        corrected = slope * y_raw + intercept
        mae_before = np.mean(np.abs(y_raw - y_true))
        mae_after  = np.mean(np.abs(corrected - y_true))
        print(f"  Group {g} ({GROUP_NAMES[g]:15s}): n={n:3d}  "
              f"slope={slope:.3f}  intercept={intercept:+.3f}  "
              f"MAE {mae_before:.3f}→{mae_after:.3f} g/dL  r={r:.3f}")
    return correction


def apply_skin_tone_correction(hb_raw: float, skin_group: int,
                                correction: dict) -> float:
    """Apply the fitted group correction to a raw Hb prediction."""
    slope, intercept = correction.get(skin_group, (1.0, 0.0))
    corrected = slope * hb_raw + intercept
    return float(np.clip(corrected, 6.0, 20.0))


def plot_skin_tone_distribution(feat_df: pd.DataFrame, path: str):
    """Plot MI distribution and group boundaries."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    mi = feat_df["mi_530_940"].values
    groups = feat_df["skin_group"].values
    colors = {1: "#F5C5A3", 2: "#C87941", 3: "#5C3317"}

    # Histogram coloured by group
    for g in [1, 2, 3]:
        axes[0].hist(mi[groups == g], bins=20, alpha=0.7,
                     color=colors[g], label=GROUP_NAMES[g], edgecolor="white")
    axes[0].axvline(SKIN_THR_LOW,  color="black", linestyle="--", lw=1.5,
                    label=f"Thr_low={SKIN_THR_LOW:.2f}")
    axes[0].axvline(SKIN_THR_HIGH, color="black", linestyle="-.", lw=1.5,
                    label=f"Thr_high={SKIN_THR_HIGH:.2f}")
    axes[0].set_xlabel("Melanin Index (DC_530/DC_940)", fontsize=10)
    axes[0].set_ylabel("Count", fontsize=10)
    axes[0].set_title("530nm Melanin Index Distribution", fontsize=11, fontweight="bold")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.2)

    # Bar chart: count per group vs mean Hb
    grp_counts = [np.sum(groups == g) for g in [1, 2, 3]]
    grp_hb     = [feat_df.loc[groups == g, "hb_true"].mean() for g in [1, 2, 3]]
    x = np.arange(3)
    bars = axes[1].bar(x, grp_counts, color=[colors[g] for g in [1,2,3]],
                       edgecolor="white", width=0.5)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([GROUP_NAMES[g] for g in [1,2,3]], fontsize=9)
    axes[1].set_ylabel("Subject count", fontsize=10)
    axes[1].set_title("Subjects per Skin-Tone Group", fontsize=11, fontweight="bold")
    ax2 = axes[1].twinx()
    ax2.plot(x, grp_hb, "D--", color="#2E75B6", lw=2, ms=8, label="Mean Hb")
    ax2.set_ylabel("Mean CBC Hb (g/dL)", fontsize=10, color="#2E75B6")
    ax2.legend(loc="upper right", fontsize=8)
    for bar, cnt in zip(bars, grp_counts):
        axes[1].text(bar.get_x() + bar.get_width()/2, cnt + 0.5,
                     str(cnt), ha="center", fontsize=10, fontweight="bold")
    axes[1].grid(True, alpha=0.2, axis="y")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_per_group_accuracy(feat_df: pd.DataFrame,
                             pred_raw_col: str,
                             pred_corr_col: str,
                             path: str):
    """Bland-Altman per skin-tone group: before vs after correction."""
    colors = {1: "#F5A623", 2: "#C87941", 3: "#5C3317"}
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for idx, g in enumerate([1, 2, 3]):
        mask   = feat_df["skin_group"] == g
        y_true = feat_df.loc[mask, "hb_true"].values
        y_raw  = feat_df.loc[mask, pred_raw_col].values
        y_corr = feat_df.loc[mask, pred_corr_col].values
        mean_r = (y_raw  + y_true) / 2
        mean_c = (y_corr + y_true) / 2
        diff_r = y_raw  - y_true
        diff_c = y_corr - y_true
        ax = axes[idx]
        ax.scatter(mean_r, diff_r, color="grey", alpha=0.5, s=25, label="Before correction")
        ax.scatter(mean_c, diff_c, color=colors[g], alpha=0.8, s=25, label="After correction")
        for diff, col, style in [(diff_r, "grey", "--"), (diff_c, colors[g], "-")]:
            bias = np.mean(diff)
            loa  = 1.96 * np.std(diff)
            ax.axhline(bias, color=col, lw=1.2, ls=style)
            ax.axhline(bias + loa, color=col, lw=0.8, ls=style, alpha=0.6)
            ax.axhline(bias - loa, color=col, lw=0.8, ls=style, alpha=0.6)
        ax.axhline(0, color="black", lw=0.6, alpha=0.4)
        ax.axhline( WHO_THRESHOLD_GDPL, color="green", lw=0.7, ls=":")
        ax.axhline(-WHO_THRESHOLD_GDPL, color="green", lw=0.7, ls=":")
        mae_raw  = np.mean(np.abs(diff_r))
        mae_corr = np.mean(np.abs(diff_c))
        w1_raw   = np.mean(np.abs(diff_r)  <= 1.0) * 100
        w1_corr  = np.mean(np.abs(diff_c) <= 1.0) * 100
        ax.set_title(f"{GROUP_NAMES[g]}\n"
                     f"MAE: {mae_raw:.2f}→{mae_corr:.2f}  "
                     f"±1g: {w1_raw:.0f}%→{w1_corr:.0f}%",
                     fontsize=10, fontweight="bold")
        ax.set_xlabel("Mean Hb (g/dL)", fontsize=9)
        ax.set_ylabel("Difference (g/dL)", fontsize=9)
        ax.legend(fontsize=7)
        ax.grid(True, alpha=0.2)
    plt.suptitle("Skin-Tone Adaptive Correction — Bland-Altman per Group",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


# =============================================================================
# CELL 5 — Hb Distribution Diagnostic
# =============================================================================

def plot_hb_distribution():
    hb_vals = np.array(list(SUBJECT_HB.values()), dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(hb_vals, bins=20, color="#2E4E8E", edgecolor="white", linewidth=0.5)
    axes[0].axvline(12.0, color="red", linestyle="--", linewidth=1.5,
                    label="Anemia threshold (♀ 12 g/dL)")
    axes[0].axvline(13.0, color="orange", linestyle="--", linewidth=1.5,
                    label="Anemia threshold (♂ 13 g/dL)")
    axes[0].set_xlabel("CBC Hb (g/dL)", fontsize=10)
    axes[0].set_ylabel("Count", fontsize=10)
    axes[0].set_title("Hb Label Distribution", fontsize=11, fontweight="bold")
    axes[0].legend(fontsize=8)
    axes[0].grid(True, alpha=0.2)
    anemic = np.sum(hb_vals < 12.0)
    normal = np.sum(hb_vals >= 12.0)
    axes[1].bar(["Anemic (<12)", "Normal (≥12)"], [anemic, normal],
                color=["#e03131", "#22a45d"], edgecolor="white")
    axes[1].set_title("Class Balance", fontsize=11, fontweight="bold")
    axes[1].set_ylabel("Count", fontsize=10)
    for i, v in enumerate([anemic, normal]):
        axes[1].text(i, v + 1, str(v), ha="center", fontsize=11, fontweight="bold")
    axes[1].grid(True, alpha=0.2, axis="y")
    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "hb_distribution.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")
    print(f"  Hb range : {hb_vals.min():.1f} – {hb_vals.max():.1f} g/dL")
    print(f"  Mean ± SD: {hb_vals.mean():.1f} ± {hb_vals.std():.1f}")
    print(f"  Anemic (<12 g/dL): {anemic} subjects ({100*anemic/len(hb_vals):.1f}%)")


# =============================================================================
# CELL 6 — Signal Processing
# =============================================================================

def _butter_bp(x, low=0.5, high=4.0, order=4):
    nyq = FS / 2
    b, a = butter(order, [low / nyq, high / nyq], btype="band")
    return filtfilt(b, a, x)

def _butter_lp(x, cutoff=0.3, order=4):
    nyq = FS / 2
    b, a = butter(order, cutoff / nyq, btype="low")
    return filtfilt(b, a, x)

def _butter_hp(x, cutoff=0.05, order=2):
    nyq = FS / 2
    b, a = butter(order, cutoff / nyq, btype="high")
    return filtfilt(b, a, x)

def _notch(x, freq=50.0, Q=30.0):
    b, a = iirnotch(freq / (FS / 2), Q)
    return filtfilt(b, a, x)

def preprocess_window(win_df: pd.DataFrame) -> dict:
    """
    Process one 8-second window for all 4 channels.
    w530 column holds the 530nm (device) / 730nm (dataset) data.
    All feature labels use 'w530' consistently.
    """
    out = {}
    for lbl in WL_LABELS:
        raw = win_df[lbl].values.astype(float)
        raw_corrected = raw * WL_CORRECTION[lbl]
        dn = _notch(raw_corrected)
        out[f"dc_{lbl}"] = _butter_lp(dn)
        out[f"ac_{lbl}"] = _butter_bp(dn)
        out[f"bl_{lbl}"] = _butter_hp(dn)
        out[f"raw_{lbl}"] = dn
    return out

def compute_sqi(proc: dict, relaxed: bool = False) -> dict:
    ac  = proc["ac_w660"]
    dc  = proc["dc_w660"]
    pi  = np.sqrt(np.mean(ac ** 2)) / (np.mean(np.abs(dc)) + 1e-10) * 100
    snr = 10 * np.log10(np.var(ac) / (np.var(ac - _butter_bp(ac)) + 1e-10))
    peaks, props = find_peaks(ac, distance=int(0.4 * FS),
                              height=0.25 * np.max(np.abs(ac) + 1e-10),
                              prominence=0.15 * np.max(np.abs(ac) + 1e-10))
    if relaxed:
        valid = (pi >= 0.03) and (snr >= 1.5) and (len(peaks) >= 2)
    else:
        valid = (pi >= 0.15) and (snr >= 4.0) and (len(peaks) >= 3)
    return {"pi": pi, "snr": snr, "n_peaks": len(peaks),
            "peaks": peaks, "valid": valid}


# =============================================================================
# CELL 7 — Feature Extraction
# Updated: all w730 references → w530; new melanin/skin-tone features added.
# =============================================================================

def extract_features(proc: dict, peaks: np.ndarray) -> dict:
    feats = {}

    # ── 1. AC/DC Ratios (Beer-Lambert basis) ────────────────────────────────
    def norm_ac(lbl):
        rms = np.sqrt(np.mean(proc[f"ac_{lbl}"] ** 2))
        dc  = np.mean(np.abs(proc[f"dc_{lbl}"])) + 1e-10
        return rms / dc

    r660  = norm_ac("w660")
    r530  = norm_ac("w530")   # ← was r730
    r850  = norm_ac("w850")
    r940  = norm_ac("w940")

    feats["R_660_940"]  = r660 / (r940  + 1e-10)
    feats["R_530_940"]  = r530 / (r940  + 1e-10)   # ← was R_730_940
    feats["R_850_940"]  = r850 / (r940  + 1e-10)
    feats["R_660_530"]  = r660 / (r530  + 1e-10)   # ← was R_660_730
    feats["R_660_850"]  = r660 / (r850  + 1e-10)
    feats["R_530_850"]  = r530 / (r850  + 1e-10)   # ← was R_730_850

    feats["R_660_940_sq"] = feats["R_660_940"] ** 2
    feats["R_850_940_sq"] = feats["R_850_940"] ** 2

    # ── 2. AC/DC values per wavelength ──────────────────────────────────────
    for lbl in WL_LABELS:
        ac_rms = np.sqrt(np.mean(proc[f"ac_{lbl}"] ** 2))
        dc_mn  = np.mean(np.abs(proc[f"dc_{lbl}"])) + 1e-10
        feats[f"pi_{lbl}"]  = ac_rms / dc_mn * 100
        feats[f"abs_{lbl}"] = -np.log(dc_mn / (dc_mn + ac_rms + 1e-10) + 1e-10)

    feats["mROA_660_940"] = feats["abs_w660"] / (feats["abs_w940"] + 1e-10)
    feats["mROA_530_940"] = feats["abs_w530"] / (feats["abs_w940"] + 1e-10)  # ← was mROA_730_940
    feats["mROA_850_940"] = feats["abs_w850"] / (feats["abs_w940"] + 1e-10)
    feats["mROA_660_530"] = feats["abs_w660"] / (feats["abs_w530"] + 1e-10)  # ← was mROA_660_730

    # ── 3. DC domain features ────────────────────────────────────────────────
    dc660 = np.mean(np.abs(proc["dc_w660"]))
    dc530 = np.mean(np.abs(proc["dc_w530"]))   # ← was dc730
    dc850 = np.mean(np.abs(proc["dc_w850"]))
    dc940 = np.mean(np.abs(proc["dc_w940"]))

    feats["dc_ratio_660_940"] = dc660 / (dc940 + 1e-10)
    feats["dc_ratio_530_940"] = dc530 / (dc940 + 1e-10)   # ← was dc_ratio_730_940
    feats["dc_ratio_850_940"] = dc850 / (dc940 + 1e-10)
    feats["dc_ratio_660_530"] = dc660 / (dc530 + 1e-10)   # ← was dc_ratio_660_730

    # ── SKIN-TONE FEATURES (530nm melanin-specific) ──────────────────────────
    # Physical basis: 530nm has ~3× higher melanin absorption than 730nm.
    # These features directly quantify melanin content independent of Hb.

    # Melanin Index: higher → darker skin
    mi_530_940 = dc530 / (dc940 + 1e-10)
    feats["mi_530_940"]  = mi_530_940
    feats["mi_660_940"]  = dc660 / (dc940 + 1e-10)

    # ITA proxy (Individual Typology Angle): log ratio of green to red DC
    # Positive = lighter skin, negative = darker skin
    feats["ita_proxy"]   = np.log(dc660 / (dc530 + 1e-10) + 1e-10)

    # Skin-tone group (1=light, 2=medium, 3=dark) — added as numeric feature
    feats["skin_group_feat"] = float(classify_skin_tone(mi_530_940))

    # 530nm AC signal variability — melanin suppresses AC amplitude at 530nm
    ac530_rms = np.sqrt(np.mean(proc["ac_w530"] ** 2))
    ac660_rms = np.sqrt(np.mean(proc["ac_w660"] ** 2))
    feats["ac_ratio_530_660"]   = ac530_rms / (ac660_rms + 1e-10)

    # DC spectral shape across [530, 660, 850, 940] nm
    # Updated wavelength array to reflect physical device
    wl_arr  = np.array([530., 660., 850., 940.])
    dc_arr  = np.array([dc530, dc660, dc850, dc940])
    dc_norm = dc_arr / (dc_arr.max() + 1e-10)
    coeffs  = np.polyfit(wl_arr, dc_norm, 2)
    feats["spectral_slope"]   = coeffs[0]
    feats["spectral_linear"]  = coeffs[1]
    feats["spectral_curve"]   = np.polyval(coeffs, 800) - np.polyval(coeffs, 700)

    # ── 4. Pulse morphology ──────────────────────────────────────────────────
    ac660 = proc["ac_w660"]
    ac850 = proc["ac_w850"]

    if len(peaks) >= 2:
        ibi = np.diff(peaks) / FS
        feats["hr_bpm"]    = 60 / (np.mean(ibi) + 1e-10)
        feats["hrv_sdnn"]  = np.std(ibi) * 1000
        feats["hrv_rmssd"] = np.sqrt(np.mean(np.diff(ibi) ** 2)) * 1000 \
                             if len(ibi) > 1 else np.nan
        pw50_list = []
        for pk in peaks:
            half  = 0.5 * ac660[pk]
            left  = np.where(ac660[:pk] < half)[0]
            right = np.where(ac660[pk:] < half)[0]
            if len(left) and len(right):
                pw50_list.append((right[0] + pk - left[-1]) / FS * 1000)
        feats["pulse_width_50"] = np.median(pw50_list) if pw50_list else np.nan
        ph = ac660[peaks]
        feats["aug_index_proxy"] = ph[1] / (ph[0] + 1e-10) if len(ph) >= 2 else np.nan
        auc_list = []
        for i in range(len(peaks) - 1):
            seg = ac660[peaks[i]:peaks[i+1]]
            auc_list.append(np.trapz(np.abs(seg)) / FS)
        feats["pulse_auc"] = np.median(auc_list) if auc_list else np.nan
        rise_list = []
        for pk in peaks:
            trough_region = ac660[max(0, pk-50):pk]
            if len(trough_region) > 0:
                trough_idx = np.argmin(trough_region)
                rise_list.append((pk - (max(0, pk-50) + trough_idx)) / FS * 1000)
        feats["rise_time_ms"] = np.median(rise_list) if rise_list else np.nan
    else:
        for key in ["hr_bpm", "hrv_sdnn", "hrv_rmssd", "pulse_width_50",
                    "aug_index_proxy", "pulse_auc", "rise_time_ms"]:
            feats[key] = np.nan

    # ── 5. APG features ──────────────────────────────────────────────────────
    apg = np.gradient(np.gradient(ac660))
    feats["apg_std"] = np.std(apg)
    apg_peaks, _   = find_peaks(apg, distance=int(0.2*FS))
    apg_troughs, _ = find_peaks(-apg, distance=int(0.2*FS))
    if len(apg_peaks) >= 2 and len(apg_troughs) >= 1:
        a_amp = apg[apg_peaks[0]]  if len(apg_peaks)   > 0 else 1e-10
        b_amp = apg[apg_troughs[0]] if len(apg_troughs) > 0 else 0.0
        feats["apg_ba_ratio"] = b_amp / (a_amp + 1e-10)
    else:
        feats["apg_ba_ratio"] = np.nan

    # ── 6. Spectral power ────────────────────────────────────────────────────
    nperseg = min(256, len(ac660) // 2)
    freqs, psd = welch(ac660, fs=FS, nperseg=nperseg)
    cardiac_mask = (freqs >= 0.7) & (freqs <= 3.5)
    noise_mask   = (freqs > 3.5) & (freqs <= 8.0)
    total_power  = np.sum(psd) + 1e-10
    feats["cardiac_power_frac"]  = np.sum(psd[cardiac_mask]) / total_power
    feats["noise_power_frac"]    = np.sum(psd[noise_mask]) / total_power
    feats["spectral_entropy_ac"] = scipy_entropy(psd / total_power + 1e-10)

    # ── 7. Waveform shape ────────────────────────────────────────────────────
    feats["ac_skewness"]  = skew(ac660)
    feats["ac_kurtosis"]  = kurtosis(ac660)
    feats["ac_crest_660"] = np.max(np.abs(ac660)) / (np.sqrt(np.mean(ac660**2)) + 1e-10)
    feats["ac_crest_850"] = np.max(np.abs(ac850)) / (np.sqrt(np.mean(ac850**2)) + 1e-10)

    # ── 8. Cross-channel correlation ─────────────────────────────────────────
    ac530 = proc["ac_w530"]   # ← was ac730
    ac940 = proc["ac_w940"]
    corr_660_940, _ = pearsonr(ac660, ac940) if len(ac660) > 3 else (np.nan, np.nan)
    corr_660_850, _ = pearsonr(ac660, ac850) if len(ac660) > 3 else (np.nan, np.nan)
    corr_660_530, _ = pearsonr(ac660, ac530) if len(ac660) > 3 else (np.nan, np.nan)
    feats["cross_corr_660_940"] = corr_660_940
    feats["cross_corr_660_850"] = corr_660_850
    feats["cross_corr_660_530"] = corr_660_530   # ← was cross_corr_660_730

    return feats


def process_subject(df: pd.DataFrame, sid: str,
                    min_windows: int = 2) -> Optional[pd.Series]:
    N = len(df)
    win_feats = []

    for start in range(0, N - WINDOW_SAMP + 1, STEP_SAMP):
        win  = df.iloc[start:start + WINDOW_SAMP]
        proc = preprocess_window(win)
        q    = compute_sqi(proc, relaxed=False)
        if not q["valid"]:
            q = compute_sqi(proc, relaxed=True)
            if not q["valid"]:
                continue
        win_feats.append(extract_features(proc, q["peaks"]))

    if len(win_feats) < min_windows:
        print(f"  [SKIP] {sid}: only {len(win_feats)} valid windows (need ≥{min_windows})")
        return None

    win_df = pd.DataFrame(win_feats)
    med    = win_df.median()
    std_   = win_df.std().rename(lambda c: f"std_{c}")
    iqr_   = (win_df.quantile(0.75) - win_df.quantile(0.25)).rename(lambda c: f"iqr_{c}")

    agg = pd.concat([med, std_, iqr_])
    agg["n_valid_windows"] = len(win_feats)
    return agg


# =============================================================================
# CELL 8 — Build Feature Matrix
# Updated: CSV column remapping w730 → w530 at load time.
# =============================================================================

def build_feature_matrix() -> pd.DataFrame:
    all_files = os.listdir(DATA_DIR)
    csv_files = [f for f in all_files if f.endswith(".csv")]

    def sort_key(fname):
        stem = fname.replace(".csv", "")
        return int(stem) if stem.isdigit() else float("inf")

    files = sorted(csv_files, key=sort_key)
    rows  = []
    skipped_no_hb   = 0
    skipped_no_sig  = 0
    skipped_missing = 0

    print(f"Processing {len(files)} subject files...")

    for fname in files:
        sid  = fname.replace(".csv", "")
        path = os.path.join(DATA_DIR, fname)

        if sid in EXCLUDE:
            print(f"  [SKIP] {sid}: excluded")
            continue

        hb = SUBJECT_HB.get(sid)
        if hb is None:
            skipped_no_hb += 1
            continue

        if os.path.getsize(path) < 100:
            print(f"  [SKIP] {sid}: empty file")
            continue

        raw_df  = pd.read_csv(path)
        col_map = {}
        for col in raw_df.columns:
            s = col.replace(" ", "").replace("nm", "").replace("_", "")
            if   s == "730": col_map[col] = "w530"   # ← remap 730nm → w530 label
            elif s == "660": col_map[col] = "w660"
            elif s == "850": col_map[col] = "w850"
            elif s == "940": col_map[col] = "w940"
        raw_df = raw_df.rename(columns=col_map)

        # Use WL_LABELS (which now has w530 not w730)
        missing_cols = [lbl for lbl in WL_LABELS if lbl not in raw_df.columns]
        if missing_cols:
            print(f"  [SKIP] {sid}: missing columns {missing_cols}")
            skipped_missing += 1
            continue

        df   = raw_df[WL_LABELS]
        feat = process_subject(df, sid)
        if feat is None:
            skipped_no_sig += 1
            continue

        feat["subject_id"]  = sid
        feat["hb_true"]     = hb
        feat["fitzpatrick"] = SUBJECT_FITZPATRICK.get(sid)

        # Add skin group derived from 530nm melanin index
        mi  = feat.get("mi_530_940", np.nan)
        feat["skin_group"] = float(classify_skin_tone(mi) if np.isfinite(mi) else 2)

        demo = get_demo_features(sid)
        for k, v in demo.items():
            feat[k] = v

        rows.append(feat)

    print(f"\n  Skipped — no Hb label : {skipped_no_hb}")
    print(f"  Skipped — poor signal : {skipped_no_sig}")
    print(f"  Skipped — missing cols: {skipped_missing}")

    if not rows:
        raise RuntimeError("Feature matrix empty — check signal processing and labels.")

    out = pd.DataFrame(rows).reset_index(drop=True)
    print(f"\n✓ Feature matrix: {out.shape[0]} subjects × {out.shape[1]} columns")
    return out


# =============================================================================
# CELL 9 — Feature Selection Utilities (unchanged)
# =============================================================================

def check_vif(X_df: pd.DataFrame, threshold: float = 10.0) -> list:
    cols = list(X_df.columns)
    dropped = []
    while True:
        vif = pd.Series(
            [variance_inflation_factor(X_df[cols].values, i) for i in range(len(cols))],
            index=cols
        )
        max_vif = vif.max()
        if max_vif < threshold:
            break
        worst = vif.idxmax()
        cols.remove(worst)
        dropped.append(worst)
    if dropped:
        print(f"  VIF removal: dropped {len(dropped)} multicollinear features")
    return cols


def select_features_rfecv(X: np.ndarray, y: np.ndarray,
                           feat_cols: list) -> tuple:
    gb_base = GradientBoostingRegressor(n_estimators=100, max_depth=3,
                                        min_samples_leaf=5, random_state=42)
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    rfecv = RFECV(estimator=gb_base, step=1, cv=cv,
                  scoring="neg_root_mean_squared_error",
                  min_features_to_select=5, n_jobs=-1)
    scaler  = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    rfecv.fit(X_scaled, y)
    selected = rfecv.support_
    sel_cols  = [c for c, s in zip(feat_cols, selected) if s]
    print(f"  RFECV selected {selected.sum()} / {len(feat_cols)} features")
    return selected, sel_cols


def add_interaction_features(X_df: pd.DataFrame,
                              top_features: list, degree: int = 2) -> pd.DataFrame:
    ratio_feats = [f for f in top_features
                   if f.startswith(("R_", "mROA_", "dc_ratio_", "mi_"))]
    ratio_feats = ratio_feats[:6]
    if len(ratio_feats) < 2:
        return X_df
    poly  = PolynomialFeatures(degree=degree, interaction_only=True, include_bias=False)
    sub   = X_df[ratio_feats].values
    inter = poly.fit_transform(sub)[:, len(ratio_feats):]
    names = [f"INTER_{'_x_'.join([ratio_feats[int(i)] for i, v in enumerate(p) if v > 0])}"
             for p in poly.powers_[len(ratio_feats):]]
    inter_df = pd.DataFrame(inter, columns=names, index=X_df.index)
    return pd.concat([X_df, inter_df], axis=1)


# =============================================================================
# CELL 10-12 — GBT Model (unchanged, used for feature selection)
# =============================================================================

def get_gradient_boosting_model() -> Pipeline:
    return Pipeline([
        ("scaler", StandardScaler()),
        ("model", GradientBoostingRegressor(
            n_estimators=500, max_depth=5, learning_rate=0.03,
            subsample=0.75, min_samples_leaf=3, max_features=0.5,
            random_state=42,
        ))
    ])


def tune_gradient_boosting(X: np.ndarray, y: np.ndarray) -> Pipeline:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    print("  Tuning Gradient Boosting...")
    gb_params = {
        "model__n_estimators":     randint(300, 800),
        "model__max_depth":        randint(3, 7),
        "model__learning_rate":    uniform(0.02, 0.08),
        "model__subsample":        uniform(0.6, 0.35),
        "model__min_samples_leaf": randint(2, 6),
        "model__max_features":     uniform(0.4, 0.5),
    }
    gb_gs = RandomizedSearchCV(
        Pipeline([("scaler", StandardScaler()),
                  ("model", GradientBoostingRegressor(random_state=42))]),
        gb_params, n_iter=30, cv=cv,
        scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1
    )
    gb_gs.fit(X, y)
    print(f"    Best params : {gb_gs.best_params_}")
    print(f"    CV RMSE     : {-gb_gs.best_score_:.4f} g/dL")
    return gb_gs.best_estimator_


def run_cv_gradient_boosting(X: np.ndarray, y: np.ndarray,
                              tuned_model: Pipeline = None) -> dict:
    cv = KFold(n_splits=5, shuffle=True, random_state=42)
    model = tuned_model if tuned_model is not None else get_gradient_boosting_model()
    print("\n── 5-Fold Cross-Validation for Gradient Boosting ─────────────────")
    y_pred   = cross_val_predict(model, X, y, cv=cv)
    diff     = y_pred - y
    rmse     = np.sqrt(mean_squared_error(y, y_pred))
    mae      = mean_absolute_error(y, y_pred)
    r2       = r2_score(y, y_pred)
    ba_bias  = np.mean(diff)
    ba_sd    = np.std(diff)
    loa_lo   = ba_bias - 1.96 * ba_sd
    loa_hi   = ba_bias + 1.96 * ba_sd
    within1  = np.mean(np.abs(diff) <= WHO_THRESHOLD_GDPL) * 100
    results  = {"y_pred": y_pred, "RMSE": rmse, "MAE": mae, "R2": r2,
                "BA_bias": ba_bias, "BA_sd": ba_sd,
                "LOA_lo": loa_lo, "LOA_hi": loa_hi, "within_1": within1}
    r2_s  = "✓ R²>0.8" if r2 >= 0.8 else ("△ R²>0.6" if r2 >= 0.6 else "✗ R²<0.6")
    who_s = "✓ WHO" if within1 >= 70 else "✗ WHO"
    print(f"  {'GBT':<22} RMSE={rmse:.3f} MAE={mae:.3f} R²={r2:.3f} "
          f"bias={ba_bias:+.3f} ±1g={within1:.1f}%  {r2_s} {who_s}")
    model.fit(X, y)
    r2_tr = r2_score(y, model.predict(X))
    print(f"    Training R²={r2_tr:.3f} (gap={r2_tr-r2:+.3f} "
          f"{'← overfit' if r2_tr-r2 > 0.3 else 'OK'})")
    return results, model


# =============================================================================
# CELL 13-16 — Plots and tables (updated for skin-tone features)
# =============================================================================

def calibrate_predictions(y_pred: np.ndarray, y_true: np.ndarray) -> np.ndarray:
    ir = IsotonicRegression(out_of_bounds="clip")
    ir.fit(y_pred, y_true)
    return ir.predict(y_pred)


def feature_correlation_report(feat_df: pd.DataFrame, available: list) -> pd.DataFrame:
    y = feat_df["hb_true"].values.astype(float)
    rows = []
    for col in available:
        x = feat_df[col].values.astype(float)
        mask = np.isfinite(x) & np.isfinite(y)
        if mask.sum() < 5:
            r, p = np.nan, np.nan
        else:
            r, p = pearsonr(x[mask], y[mask])
        rows.append({"feature": col, "pearson_r": round(r, 3), "p_value": round(p, 4)})
    corr_df = pd.DataFrame(rows).sort_values("pearson_r", key=abs, ascending=False)
    print("\n── Feature–Hb Pearson Correlation (Top 20) ──────────────────────")
    print(corr_df.head(20).to_string(index=False))
    skin_feats = [r for r in rows if "530" in r["feature"] or "skin" in r["feature"] or "ita" in r["feature"] or "mi_" in r["feature"]]
    print("\n── 530nm Skin-Tone Feature Correlations ─────────────────────────")
    for r in sorted(skin_feats, key=lambda x: abs(x["pearson_r"]), reverse=True):
        print(f"  {r['feature']:<30}  r={r['pearson_r']:+.3f}  p={r['p_value']:.4f}")
    return corr_df


def feature_importance_gradient_boosting(X: np.ndarray, y: np.ndarray,
                                          feat_cols: list, model: Pipeline) -> pd.DataFrame:
    gb_model = model.named_steps["model"]
    return pd.DataFrame({
        "feature":    feat_cols,
        "importance": gb_model.feature_importances_
    }).sort_values("importance", ascending=False).reset_index(drop=True)


def plot_bland_altman(results: dict, y: np.ndarray, path: str):
    fig, ax = plt.subplots(figsize=(8, 6))
    diff = results["y_pred"] - y
    mean_val = (results["y_pred"] + y) / 2
    ax.scatter(mean_val, diff, color="#e03131", alpha=0.6, s=30,
               edgecolors="white", linewidth=0.3)
    ax.axhline(results["BA_bias"], color="#e03131", linewidth=2,
               label=f"Bias={results['BA_bias']:+.3f}")
    ax.axhline(results["LOA_lo"], color="#e03131", linewidth=1.2, linestyle="--",
               label=f"LOA [{results['LOA_lo']:.2f},{results['LOA_hi']:.2f}]")
    ax.axhline(results["LOA_hi"], color="#e03131", linewidth=1.2, linestyle="--")
    ax.axhline(0, color="black", linewidth=0.8, alpha=0.4)
    ax.axhline( WHO_THRESHOLD_GDPL, color="grey", linewidth=0.8, linestyle=":")
    ax.axhline(-WHO_THRESHOLD_GDPL, color="grey", linewidth=0.8, linestyle=":")
    ax.set_title(f"Gradient Boosting\n≤1g/dL={results['within_1']:.1f}%",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Mean Hb (g/dL)", fontsize=10)
    ax.set_ylabel("Difference (g/dL)", fontsize=10)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_scatter(results: dict, y: np.ndarray, path: str):
    fig, ax = plt.subplots(figsize=(8, 6))
    lo, hi = y.min() - 0.5, y.max() + 0.5
    ax.scatter(y, results["y_pred"], color="#e03131", alpha=0.6, s=30,
               edgecolors="white", linewidth=0.3)
    ax.plot([lo, hi], [lo, hi], "k--", linewidth=1, alpha=0.5)
    ax.plot([lo, hi], [lo + WHO_THRESHOLD_GDPL, hi + WHO_THRESHOLD_GDPL],
            color="grey", linewidth=0.8, linestyle=":", alpha=0.5)
    ax.plot([lo, hi], [lo - WHO_THRESHOLD_GDPL, hi - WHO_THRESHOLD_GDPL],
            color="grey", linewidth=0.8, linestyle=":", alpha=0.5)
    ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
    ax.set_title("GBT: Predicted vs True Hb", fontsize=11, fontweight="bold")
    ax.set_xlabel("True Hb — CBC (g/dL)", fontsize=10)
    ax.set_ylabel("Predicted Hb (g/dL)", fontsize=10)
    ax.text(0.05, 0.88,
            f"RMSE={results['RMSE']:.3f}\nMAE={results['MAE']:.3f}\nR²={results['R2']:.3f}",
            transform=ax.transAxes, fontsize=10,
            bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.8))
    ax.grid(True, alpha=0.2)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_feature_importance(imp_df: pd.DataFrame, path: str, top_n: int = 20):
    top = imp_df.head(top_n)
    colors = []
    for f in top["feature"]:
        if any(k in f for k in ["530", "mi_", "ita_", "skin_", "ac_ratio_530"]):
            colors.append("#FF6B35")          # skin-tone features — orange
        elif any(k in f for k in ["mROA", "abs_", "R_660", "R_530"]):
            colors.append("#e03131")
        elif any(k in f for k in ["dc_ratio", "spectral"]):
            colors.append("#e8a317")
        elif any(k in f for k in ["pulse", "rise", "apg", "aug"]):
            colors.append("#22a45d")
        elif any(k in f for k in ["std_", "iqr_"]):
            colors.append("#7B5EA7")
        elif f.startswith("demo_"):
            colors.append("#2E4E8E")
        else:
            colors.append("#555555")
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.barh(top["feature"][::-1], top["importance"][::-1],
            color=colors[::-1], edgecolor="white", linewidth=0.5)
    ax.set_xlabel("Feature Importance (Gini)", fontsize=10)
    ax.set_title(f"GBT Feature Importance — Top {top_n}\n"
                 f"(530nm skin-tone features highlighted in orange)",
                 fontsize=11, fontweight="bold")
    ax.grid(True, axis="x", alpha=0.3)
    legend = [
        Patch(color="#FF6B35", label="Skin-tone / melanin (530nm)"),
        Patch(color="#e03131", label="Beer-Lambert / mROA ratios"),
        Patch(color="#e8a317", label="DC spectral"),
        Patch(color="#22a45d", label="Pulse morphology"),
        Patch(color="#7B5EA7", label="Window variability (std/IQR)"),
        Patch(color="#2E4E8E", label="Demographic"),
        Patch(color="#555555", label="Other"),
    ]
    ax.legend(handles=legend, fontsize=8, loc="lower right")
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def plot_correlation_bar(corr_df: pd.DataFrame, path: str):
    fig, ax = plt.subplots(figsize=(11, 8))
    colors = ["#22a45d" if r > 0 else "#e03131" for r in corr_df["pearson_r"]]
    ax.barh(corr_df["feature"][::-1], corr_df["pearson_r"][::-1],
            color=colors[::-1], edgecolor="white", linewidth=0.4)
    ax.axvline(0,   color="black", linewidth=0.8)
    ax.axvline( 0.3, color="grey", linewidth=0.8, linestyle="--", alpha=0.6,
               label="|r| = 0.3 (useful)")
    ax.axvline(-0.3, color="grey", linewidth=0.8, linestyle="--", alpha=0.6)
    ax.set_xlabel("Pearson r with CBC Hb", fontsize=10)
    ax.set_title("Feature–Hb Correlation", fontsize=11, fontweight="bold")
    ax.legend(fontsize=8)
    ax.grid(True, axis="x", alpha=0.2)
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"  Saved: {path}")


def save_results_table(results: dict, path: str) -> pd.DataFrame:
    row = {
        "Model":              "Gradient Boosting",
        "RMSE (g/dL)":        round(results["RMSE"],    3),
        "MAE (g/dL)":         round(results["MAE"],     3),
        "R²":                 round(results["R2"],      3),
        "BA Bias (g/dL)":     round(results["BA_bias"], 3),
        "LOA Lower":          round(results["LOA_lo"],  3),
        "LOA Upper":          round(results["LOA_hi"],  3),
        "Within ±1 g/dL (%)": round(results["within_1"], 1),
        "WHO Pass (≥70%)":    "Yes" if results["within_1"] >= 70 else "No",
    }
    df = pd.DataFrame([row])
    df.to_csv(path, index=False)
    print(f"  Saved: {path}")
    return df


# =============================================================================
# CELL 17 — MAIN
# =============================================================================

if __name__ == "__main__":
    print("=" * 70)
    print("  Development of An Affordable Non-Invasive Anemia Screening Device")
    print("  Using Multi-Wavelength Optical Sensing, Edge ML &")
    print("  Adaptive Skin-Tone Adaptation")
    print("=" * 70)

    print("\n── Hb Label Distribution ────────────────────────────────────────")
    plot_hb_distribution()

    feat_df = build_feature_matrix()

    # ── Auto-compute melanin thresholds from data ─────────────────────────
    print("\n── Skin-Tone Classification from 530nm ──────────────────────────")
    thr_low, thr_high = compute_melanin_thresholds(feat_df)
    SKIN_THR_LOW  = thr_low
    SKIN_THR_HIGH = thr_high
    # Re-classify with data-driven thresholds
    feat_df["skin_group"] = feat_df["mi_530_940"].apply(classify_skin_tone)
    grp_counts = feat_df["skin_group"].value_counts().sort_index()
    for g, n in grp_counts.items():
        mean_hb = feat_df.loc[feat_df["skin_group"] == g, "hb_true"].mean()
        print(f"  Group {g} ({GROUP_NAMES[g]:15s}): {n:3d} subjects  mean Hb={mean_hb:.2f} g/dL")
    plot_skin_tone_distribution(feat_df,
                                os.path.join(OUTPUT_DIR, "skin_tone_distribution.png"))

    meta_cols = {"subject_id", "hb_true", "fitzpatrick",
                 "n_valid_windows", "skin_group"}
    all_feat_cols = [c for c in feat_df.columns if c not in meta_cols]

    # Demographic coverage
    demo_names = [c for c in all_feat_cols if c.startswith("demo_")]
    print("\n── Demographic Feature Coverage ─────────────────────────────────")
    for dn in demo_names:
        n_ok  = feat_df[dn].notna().sum()
        n_tot = len(feat_df)
        r_val, _ = pearsonr(
            feat_df[dn].fillna(feat_df[dn].median()),
            feat_df["hb_true"]
        ) if n_ok > 5 else (float("nan"), None)
        print(f"  {dn:<25}  {n_ok:>3}/{n_tot}  r(Hb)={r_val:+.3f}")

    nan_frac  = feat_df[all_feat_cols].isna().mean()
    available = [c for c in all_feat_cols
                 if nan_frac[c] <= 0.3 or c.startswith("demo_")]
    dropped   = [c for c in all_feat_cols
                 if nan_frac[c] > 0.3 and not c.startswith("demo_")]
    if dropped:
        print(f"\n  Dropping high-NaN PPG features ({len(dropped)}): {dropped[:5]}...")

    feat_df[available] = feat_df[available].fillna(feat_df[available].median())

    corr_df = feature_correlation_report(feat_df, available)
    corr_df.to_csv(os.path.join(OUTPUT_DIR, "feature_correlations.csv"), index=False)
    plot_correlation_bar(corr_df, os.path.join(OUTPUT_DIR, "feature_correlations.png"))

    informative = corr_df[corr_df["pearson_r"].abs() >= 0.05]["feature"].tolist()

    # Always keep skin-tone features even if their direct Hb correlation is low.
    # These features encode melanin, not Hb — they correlate with prediction
    # bias, not Hb directly, so Pearson r with Hb will be near zero.
    SKIN_PROTECTED = ["mi_530_940", "skin_group_feat", "ita_proxy",
                      "ac_ratio_530_660", "dc_ratio_530_940"]
    protected_present = [f for f in SKIN_PROTECTED if f in available]
    informative = list(set(informative) | set(protected_present))
    print(f"\n  Informative features (|r| ≥ 0.05 + skin-tone protected): "
          f"{len(informative)} / {len(available)}")
    available = [c for c in available if c in informative]

    feat_df.to_csv(os.path.join(OUTPUT_DIR, "feature_matrix.csv"), index=False)

    top10 = corr_df.head(10)["feature"].tolist()
    feat_df_aug = add_interaction_features(feat_df[available], top10)
    available_aug = list(feat_df_aug.columns)

    X_full = feat_df_aug.values
    y      = feat_df["hb_true"].values.astype(float)

    print("\n── VIF Multicollinearity Check ──────────────────────────────────")
    try:
        X_df_vif = pd.DataFrame(X_full, columns=available_aug)
        X_df_vif_std = pd.DataFrame(
            StandardScaler().fit_transform(X_df_vif), columns=available_aug)
        vif_cols = check_vif(X_df_vif_std, threshold=15.0)
        X_full = X_df_vif_std[vif_cols].values
        available_aug = vif_cols
        print(f"  Features after VIF: {len(available_aug)}")
    except Exception as e:
        print(f"  VIF check skipped: {e}")

    print("\n── RFECV Feature Selection ──────────────────────────────────────")
    try:
        sel_mask, sel_cols = select_features_rfecv(X_full, y, available_aug)
        X = X_full[:, sel_mask]
        feat_cols_final = sel_cols
    except Exception as e:
        print(f"  RFECV failed ({e}), using all features")
        X = X_full
        feat_cols_final = available_aug

    print(f"\nDataset ready: {len(y)} subjects | "
          f"Hb {y.min():.1f}–{y.max():.1f} g/dL (mean {y.mean():.1f})")
    print(f"Final features: {X.shape[1]}")

    print("\n── Hyperparameter Tuning ────────────────────────────────────────")
    tuned_model = tune_gradient_boosting(X, y)

    results, final_model = run_cv_gradient_boosting(X, y, tuned_model=tuned_model)

    print("\n── Feature Importance (Gradient Boosting) ───────────────────────")
    imp_df = feature_importance_gradient_boosting(X, y, feat_cols_final, final_model)
    print(imp_df.head(15).to_string(index=False))
    imp_df.to_csv(os.path.join(OUTPUT_DIR, "feature_importance.csv"), index=False)

    # ── Skin-Tone Adaptive Correction ────────────────────────────────────────
    print("\n── Skin-Tone Adaptive Correction ────────────────────────────────")
    feat_df["hb_pred_raw"] = results["y_pred"]

    # Fit per-group linear correction
    SKIN_CORRECTION = fit_skin_tone_correction(feat_df, y_pred_col="hb_pred_raw")

    # Apply correction
    feat_df["hb_pred_corrected"] = feat_df.apply(
        lambda row: apply_skin_tone_correction(
            row["hb_pred_raw"], int(row["skin_group"]), SKIN_CORRECTION),
        axis=1
    )

    # Overall metrics after correction
    y_corr  = feat_df["hb_pred_corrected"].values
    diff_c  = y_corr - y
    rmse_c  = np.sqrt(mean_squared_error(y, y_corr))
    mae_c   = mean_absolute_error(y, y_corr)
    r2_c    = r2_score(y, y_corr)
    within1_c = np.mean(np.abs(diff_c) <= WHO_THRESHOLD_GDPL) * 100

    print(f"\n── Final Results — With Skin-Tone Adaptive Correction ───────────")
    print(f"  {'Metric':<22} {'Before':>8} {'After':>8}")
    print(f"  {'RMSE (g/dL)':<22} {results['RMSE']:>8.3f} {rmse_c:>8.3f}")
    print(f"  {'MAE (g/dL)':<22} {results['MAE']:>8.3f} {mae_c:>8.3f}")
    print(f"  {'R²':<22} {results['R2']:>8.3f} {r2_c:>8.3f}")
    print(f"  {'Within ±1 g/dL (%)':<22} {results['within_1']:>8.1f} {within1_c:>8.1f}")
    who_s = "✓ PASS" if within1_c >= 70 else "✗ FAIL"
    print(f"  {'WHO criterion':<22} {'':>8} {who_s:>8}")

    # Save correction parameters for ESP32 firmware
    correction_export = {
        "SKIN_THR_LOW":  float(SKIN_THR_LOW),
        "SKIN_THR_HIGH": float(SKIN_THR_HIGH),
        "groups": {
            str(g): {"slope": float(v[0]), "intercept": float(v[1])}
            for g, v in SKIN_CORRECTION.items()
        }
    }
    with open(os.path.join(OUTPUT_DIR, "skin_correction.json"), "w") as f:
        json.dump(correction_export, f, indent=2)
    print(f"\n  Skin correction params saved: "
          f"{os.path.join(OUTPUT_DIR, 'skin_correction.json')}")

    # ── Outputs ───────────────────────────────────────────────────────────────
    print("\n── Saving outputs ───────────────────────────────────────────────")
    plot_bland_altman(results, y, os.path.join(OUTPUT_DIR, "bland_altman.png"))
    plot_scatter(results, y, os.path.join(OUTPUT_DIR, "scatter_plots.png"))
    plot_feature_importance(imp_df,
                            os.path.join(OUTPUT_DIR, "feature_importance.png"), top_n=20)
    plot_per_group_accuracy(feat_df, "hb_pred_raw", "hb_pred_corrected",
                            os.path.join(OUTPUT_DIR, "skin_tone_ba.png"))
    save_results_table(results, os.path.join(OUTPUT_DIR, "results_summary.csv"))
    print("\nAll outputs saved to:", OUTPUT_DIR)


# =============================================================================
# CELL 18 — Random Forest + emlearn export (with skin-tone features)
# =============================================================================

!pip install emlearn   # uncomment in Colab

import emlearn
from google.colab import files

print("\n── Training Random Forest with skin-tone adaptation ─────────────────")

# ── FIX: source from feat_df (not feat_df_aug) ───────────────────────────────
# feat_df_aug has renamed/dropped columns after VIF+RFECV. The 530nm skin-tone
# features (mi_530_940, skin_group_feat, ita_proxy, ac_ratio_530_660) are
# plain median-level columns that exist in feat_df but NOT in feat_df_aug.
# Reading from feat_df directly avoids the KeyError.

top_features_candidates = [
    "demo_sex",
    "demo_age",
    "mi_530_940",          # median melanin index (530nm)
    "skin_group_feat",     # median skin group 1/2/3
    "ita_proxy",           # median ITA proxy log(DC660/DC530)
    "ac_ratio_530_660",    # median AC amplitude ratio 530/660
    "iqr_aug_index_proxy",
    "std_rise_time_ms",
    "iqr_apg_ba_ratio",
    "std_apg_ba_ratio",
    "std_R_530_850",
    "std_dc_ratio_850_940",
    "std_ac_crest_850",
    "std_aug_index_proxy",
    "pi_w660",
    "abs_w660",
]

# Keep only columns that actually exist in feat_df after all prior processing
top_features = [f for f in top_features_candidates if f in feat_df.columns]
missing      = [f for f in top_features_candidates if f not in feat_df.columns]

if missing:
    print(f"  WARNING skipped (not in feat_df): {missing}")
    avail = [c for c in feat_df.columns if "530" in c or "skin" in c or "ita" in c]
    print(f"  Available 530nm/skin cols: {avail}")
    print("  Check that extract_features() ran with WL_LABELS=['w530','w660','w850','w940']")

print(f"  Using {len(top_features)} features: {top_features}")

# Build X_reduced from feat_df directly
rf_source = feat_df[top_features].copy()
rf_source = rf_source.fillna(rf_source.median())
X_reduced = rf_source.values

rf_reduced = RandomForestRegressor(
    n_estimators=25,
    max_depth=5,
    min_samples_leaf=4,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
rf_reduced.fit(X_reduced, y)
y_pred_rf = rf_reduced.predict(X_reduced)
print(f"MAE  : {mean_absolute_error(y, y_pred_rf):.3f} g/dL")
print(f"RMSE : {np.sqrt(mean_squared_error(y, y_pred_rf)):.3f} g/dL")

# Apply skin-tone correction to RF predictions too
feat_df["hb_rf_raw"]  = y_pred_rf
feat_df["hb_rf_corr"] = feat_df.apply(
    lambda row: apply_skin_tone_correction(
        row["hb_rf_raw"], int(row["skin_group"]), SKIN_CORRECTION),
    axis=1
)
rmse_rf_corr = np.sqrt(mean_squared_error(
    y, feat_df["hb_rf_corr"].values))
w1_rf_corr   = np.mean(np.abs(feat_df["hb_rf_corr"].values - y) <= 1.0) * 100
print(f"\nRF + skin correction — RMSE={rmse_rf_corr:.3f}  ±1g={w1_rf_corr:.1f}%")

# Convert to emlearn C header
cmodel = emlearn.convert(rf_reduced, method='inline')

MODEL_DIR = "/content/esp32_final"
os.makedirs(MODEL_DIR, exist_ok=True)
model_path = os.path.join(MODEL_DIR, "hb_rf_model.h")
cmodel.save(file=model_path, name="hb_model")

size_kb = os.path.getsize(model_path) / 1024
print(f"\n✅ Skin-tone adaptive model saved!")
print(f"   Features : {len(top_features)}")
print(f"   Size     : {size_kb:.1f} KB")
print(f"   Path     : {model_path}")

# Save feature list and correction params together for ESP32 reference
export_meta = {
    "feature_names":   top_features,
    "n_features":      len(top_features),
    "skin_thresholds": correction_export,
    "model_notes":     "530nm replaces 730nm. mi_530_940 used for skin-tone group. "
                       "Apply SKIN_CORRECTION after hb_model_predict()."
}
with open(os.path.join(MODEL_DIR, "model_meta.json"), "w") as f:
    json.dump(export_meta, f, indent=2)

files.download(model_path)
files.download(os.path.join(MODEL_DIR, "model_meta.json"))
files.download(os.path.join(OUTPUT_DIR, "skin_correction.json"))
print("\n📥 Three files downloaded: hb_rf_model.h, model_meta.json, skin_correction.json")
# =============================================================================
# EVALUATION CELL — Regression + Binary Classification Performance
# =============================================================================
# Paste this cell at the END of Final.ipynb, after Cell 18 (emlearn export).
#
# It evaluates the Random Forest model (rf_reduced / skin-corrected) on:
#   A) Regression metrics     — RMSE, MAE, R², Bias, LoA, Within ±1 g/dL
#   B) Binary classification  — Anemic vs Non-Anemic using WHO thresholds
#      Metrics: Accuracy, Precision, Recall (Sensitivity), Specificity,
#               F1-Score, AUC-ROC, TP, TN, FP, FN, Confusion Matrix
#
# Anemia thresholds (WHO):
#   Male   : Hb < 13.0 g/dL → Anemic
#   Female : Hb < 12.0 g/dL → Anemic
#   Sex-blind fallback (if sex not available): Hb < 12.0 g/dL
#
# Requires: rf_reduced, feat_df, y already defined from prior cells.
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve,
    ConfusionMatrixDisplay
)
import warnings
warnings.filterwarnings("ignore")

# ── 0. Regenerate predictions ─────────────────────────────────────────────────
# Use the same X_reduced and skin correction already computed in Cell 18.
# rf_reduced was fitted on ALL 250 subjects (no held-out test set in original
# pipeline). We evaluate on the same data (training performance) AND add
# leave-one-out cross-validation for unbiased estimates.

from sklearn.model_selection import cross_val_predict, KFold

print("=" * 68)
print("  MODEL EVALUATION — Regression + Anemia Classification")
print("=" * 68)

# ── Re-build X_reduced in case cell order changed ────────────────────────────
top_features_eval = [
    "demo_sex", "demo_age", "mi_530_940", "skin_group_feat", "ita_proxy",
    "ac_ratio_530_660", "iqr_aug_index_proxy", "std_rise_time_ms",
    "iqr_apg_ba_ratio", "std_apg_ba_ratio", "std_R_530_850",
    "std_dc_ratio_850_940", "std_ac_crest_850", "std_aug_index_proxy",
    "pi_w660", "abs_w660",
]
top_features_eval = [f for f in top_features_eval if f in feat_df.columns]
X_eval  = feat_df[top_features_eval].fillna(feat_df[top_features_eval].median()).values
y_true  = feat_df["hb_true"].values.astype(float)

# Sex vector for sex-specific thresholds
sex_vec = feat_df["demo_sex"].fillna(0).values   # 1=Male, 0=Female

# ── 5-Fold CV predictions (unbiased) ─────────────────────────────────────────
cv = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred_cv = cross_val_predict(rf_reduced, X_eval, y_true, cv=cv)

# Apply skin-tone correction to CV predictions
skin_groups_eval = feat_df["skin_group"].values.astype(int)

def apply_correction_vec(y_raw, groups, correction):
    out = np.empty_like(y_raw)
    for i, (h, g) in enumerate(zip(y_raw, groups)):
        slope, intercept = correction.get(int(g), (1.0, 0.0))
        out[i] = np.clip(slope * h + intercept, 6.0, 20.0)
    return out

y_pred_cv_corr = apply_correction_vec(y_pred_cv, skin_groups_eval, SKIN_CORRECTION)

# Training predictions (for comparison — optimistic, shown separately)
y_pred_train      = rf_reduced.predict(X_eval)
y_pred_train_corr = apply_correction_vec(y_pred_train, skin_groups_eval, SKIN_CORRECTION)

# =============================================================================
# A) REGRESSION METRICS
# =============================================================================

def regression_metrics(y_true, y_pred, label=""):
    diff   = y_pred - y_true
    rmse   = np.sqrt(mean_squared_error(y_true, y_pred))
    mae    = mean_absolute_error(y_true, y_pred)
    r2     = r2_score(y_true, y_pred)
    bias   = np.mean(diff)
    sd     = np.std(diff)
    loa_lo = bias - 1.96 * sd
    loa_hi = bias + 1.96 * sd
    w1     = np.mean(np.abs(diff) <= 1.0) * 100
    pearson_r = np.corrcoef(y_true, y_pred)[0, 1]
    return {
        "label": label, "RMSE": rmse, "MAE": mae, "R2": r2,
        "Pearson_r": pearson_r, "Bias": bias, "SD": sd,
        "LoA_lo": loa_lo, "LoA_hi": loa_hi, "Within_1g": w1
    }

reg_cv    = regression_metrics(y_true, y_pred_cv_corr,   "5-Fold CV  (skin-corrected)")
reg_train = regression_metrics(y_true, y_pred_train_corr, "Training   (skin-corrected)")

print("\n── A) Regression Performance ────────────────────────────────────")
print(f"  {'Metric':<25} {'5-Fold CV':>12}  {'Training':>12}  {'Notes'}")
print(f"  {'-'*65}")
rows_reg = [
    ("RMSE  (g/dL)",    "RMSE",      ".3f", "lower is better"),
    ("MAE   (g/dL)",    "MAE",       ".3f", "lower is better"),
    ("R²",              "R2",        ".3f", "≥0.8 excellent, ≥0.6 good"),
    ("Pearson r",       "Pearson_r", ".3f", "correlation"),
    ("Bias (g/dL)",     "Bias",      "+.3f","systematic offset"),
    ("StdDev diff",     "SD",        ".3f", "random error spread"),
    ("LoA lower",       "LoA_lo",    "+.3f","Bland-Altman −1.96 SD"),
    ("LoA upper",       "LoA_hi",    "+.3f","Bland-Altman +1.96 SD"),
    ("Within ±1 g/dL",  "Within_1g", ".1f", "WHO criterion ≥70%"),
]
for name, key, fmt, note in rows_reg:
    cv_val = format(reg_cv[key],    fmt)
    tr_val = format(reg_train[key], fmt)
    print(f"  {name:<25} {cv_val:>12}  {tr_val:>12}  {note}")

who_pass = "✓ PASS" if reg_cv["Within_1g"] >= 70 else "✗ FAIL"
r2_pass  = ("✓ Excellent" if reg_cv["R2"] >= 0.8 else
            ("△ Acceptable" if reg_cv["R2"] >= 0.6 else "✗ Poor"))
print(f"\n  WHO ±1 g/dL criterion : {who_pass}  ({reg_cv['Within_1g']:.1f}%)")
print(f"  R² interpretation     : {r2_pass}")

# =============================================================================
# B) BINARY CLASSIFICATION METRICS  (Anemia Detection)
# =============================================================================

# WHO sex-specific thresholds
def anemia_labels(hb_vals, sex_vals, male_thr=13.0, female_thr=12.0):
    """1 = Anemic, 0 = Non-Anemic"""
    labels = np.zeros(len(hb_vals), dtype=int)
    for i, (h, s) in enumerate(zip(hb_vals, sex_vals)):
        thr = male_thr if s == 1 else female_thr
        labels[i] = 1 if h < thr else 0
    return labels

y_true_cls = anemia_labels(y_true,         sex_vec)
y_pred_cls = anemia_labels(y_pred_cv_corr, sex_vec)

# For AUC: use continuous predicted Hb (inverted: lower Hb = more anemic)
# We flip sign so that "higher score = more anemic" for sklearn's AUC
auc_score_val = roc_auc_score(y_true_cls, -y_pred_cv_corr)

# Confusion matrix components
cm = confusion_matrix(y_true_cls, y_pred_cls)
# cm layout: [[TN, FP], [FN, TP]]
TN, FP, FN, TP = cm.ravel() if cm.shape == (2,2) else (0,0,0,0)

accuracy    = accuracy_score(y_true_cls, y_pred_cls)
precision   = precision_score(y_true_cls, y_pred_cls, zero_division=0)
recall      = recall_score(y_true_cls, y_pred_cls, zero_division=0)   # = Sensitivity
f1          = f1_score(y_true_cls, y_pred_cls, zero_division=0)
specificity = TN / (TN + FP) if (TN + FP) > 0 else 0.0
npv         = TN / (TN + FN) if (TN + FN) > 0 else 0.0   # Negative Predictive Value
ppv         = precision   # same as precision

n_anemic     = int(y_true_cls.sum())
n_nonanemic  = int((y_true_cls == 0).sum())

print(f"\n── B) Binary Classification — Anemia Detection ─────────────────")
print(f"  Threshold: Male <13 g/dL, Female <12 g/dL (WHO)")
print(f"  Dataset  : {n_anemic} anemic  |  {n_nonanemic} non-anemic  "
      f"(total {len(y_true_cls)})")
print(f"\n  Confusion Matrix:")
print(f"                     Predicted")
print(f"                  Anemic  Non-Anemic")
print(f"  Actual Anemic      {TP:>4}    {FN:>4}      (TP | FN)")
print(f"  Actual Non-Anemic  {FP:>4}    {TN:>4}      (FP | TN)")

print(f"\n  {'Metric':<30} {'Value':>8}  Notes")
print(f"  {'-'*60}")
cls_rows = [
    ("True Positives  (TP)",     TP,           "d",    "correctly detected anemic"),
    ("True Negatives  (TN)",     TN,           "d",    "correctly detected normal"),
    ("False Positives (FP)",     FP,           "d",    "normal predicted as anemic"),
    ("False Negatives (FN)",     FN,           "d",    "anemic missed ← critical"),
    ("Accuracy",                 accuracy,     ".3f",  "overall correct rate"),
    ("Precision / PPV",          ppv,          ".3f",  "of predicted anemic, truly anemic"),
    ("Recall / Sensitivity",     recall,       ".3f",  "of true anemic, correctly found"),
    ("Specificity",              specificity,  ".3f",  "of true normal, correctly found"),
    ("F1-Score",                 f1,           ".3f",  "harmonic mean of P & R"),
    ("NPV",                      npv,          ".3f",  "of predicted normal, truly normal"),
    ("AUC-ROC",                  auc_score_val,".3f",  "≥0.9 excellent, ≥0.7 acceptable"),
]
for name, val, fmt, note in cls_rows:
    v_str = format(val, fmt)
    print(f"  {name:<30} {v_str:>8}  {note}")

# Clinical interpretation
print(f"\n  Clinical interpretation:")
if recall >= 0.90:
    print(f"  ✓ Sensitivity {recall:.1%} — high; few anemic cases missed")
elif recall >= 0.70:
    print(f"  △ Sensitivity {recall:.1%} — acceptable; some anemic cases missed")
else:
    print(f"  ✗ Sensitivity {recall:.1%} — low; too many anemic cases missed")
if specificity >= 0.90:
    print(f"  ✓ Specificity {specificity:.1%} — high; few healthy patients misclassified")
elif specificity >= 0.70:
    print(f"  △ Specificity {specificity:.1%} — acceptable")
else:
    print(f"  ✗ Specificity {specificity:.1%} — low; too many false alarms")
if FN > 0:
    print(f"  ⚠ {FN} anemic subject(s) were missed (False Negatives) — "
          f"consider lowering prediction threshold for screening use")

# =============================================================================
# C) VISUALISATIONS  (6-panel figure)
# =============================================================================

fpr_arr, tpr_arr, _ = roc_curve(y_true_cls, -y_pred_cv_corr)

fig = plt.figure(figsize=(18, 11))
fig.suptitle(
    "RF Model Evaluation — Regression & Anemia Classification\n"
    f"250 subjects | 5-Fold CV | RMSE={reg_cv['RMSE']:.3f} g/dL  "
    f"R²={reg_cv['R2']:.3f}  AUC={auc_score_val:.3f}",
    fontsize=13, fontweight="bold", y=0.98
)
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

BLUE  = "#2E4E8E"
GREEN = "#22a45d"
RED   = "#e03131"
ORANGE= "#F5A623"

# ── Panel 1: Scatter predicted vs true ───────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
scatter_c = [RED if a == 1 else BLUE for a in y_true_cls]
ax1.scatter(y_true, y_pred_cv_corr, c=scatter_c, alpha=0.55, s=28, edgecolors="none")
mn, mx = min(y_true.min(), y_pred_cv_corr.min()), max(y_true.max(), y_pred_cv_corr.max())
ax1.plot([mn, mx], [mn, mx], "k--", lw=1.2, label="Perfect")
ax1.plot([mn, mx], [mn+1, mx+1], color="grey", lw=0.8, ls=":")
ax1.plot([mn, mx], [mn-1, mx-1], color="grey", lw=0.8, ls=":", label="±1 g/dL")
ax1.set_xlabel("Actual Hb (g/dL)", fontsize=10)
ax1.set_ylabel("Predicted Hb (g/dL)", fontsize=10)
ax1.set_title(f"Scatter  R²={reg_cv['R2']:.3f}  r={reg_cv['Pearson_r']:.3f}",
              fontsize=10, fontweight="bold")
ax1.legend(fontsize=8)
from matplotlib.lines import Line2D
handles = [Line2D([0],[0],marker='o',color='w',markerfacecolor=RED,  markersize=7,label='Anemic'),
           Line2D([0],[0],marker='o',color='w',markerfacecolor=BLUE, markersize=7,label='Non-Anemic')]
ax1.legend(handles=handles, fontsize=8, loc="upper left")
ax1.grid(True, alpha=0.2)

# ── Panel 2: Bland-Altman ─────────────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
mean_vals = (y_true + y_pred_cv_corr) / 2
diff_vals = y_pred_cv_corr - y_true
ax2.scatter(mean_vals, diff_vals, c=scatter_c, alpha=0.55, s=28, edgecolors="none")
ax2.axhline(reg_cv["Bias"],   color=BLUE,  lw=1.5, label=f"Bias={reg_cv['Bias']:+.3f}")
ax2.axhline(reg_cv["LoA_hi"], color=ORANGE,lw=1.2, ls="--",
            label=f"LoA {reg_cv['LoA_lo']:+.2f} to {reg_cv['LoA_hi']:+.2f}")
ax2.axhline(reg_cv["LoA_lo"], color=ORANGE,lw=1.2, ls="--")
ax2.axhline( 1.0, color="green", lw=0.7, ls=":")
ax2.axhline(-1.0, color="green", lw=0.7, ls=":", label="±1 g/dL")
ax2.axhline(0,    color="black", lw=0.5, alpha=0.4)
ax2.set_xlabel("Mean Hb (g/dL)", fontsize=10)
ax2.set_ylabel("Difference (Predicted − Actual)", fontsize=10)
ax2.set_title("Bland-Altman Agreement Plot", fontsize=10, fontweight="bold")
ax2.legend(fontsize=7)
ax2.grid(True, alpha=0.2)

# ── Panel 3: Error distribution histogram ────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(diff_vals, bins=25, color=BLUE, edgecolor="white", alpha=0.8)
ax3.axvline(0,                color="black", lw=1.0, ls="--", label="Zero error")
ax3.axvline(reg_cv["Bias"],   color=RED,    lw=1.5, label=f"Bias={reg_cv['Bias']:+.3f}")
ax3.axvline( 1.0, color="green", lw=0.8, ls=":")
ax3.axvline(-1.0, color="green", lw=0.8, ls=":", label="±1 g/dL")
ax3.set_xlabel("Prediction Error (g/dL)", fontsize=10)
ax3.set_ylabel("Count", fontsize=10)
ax3.set_title(f"Error Distribution  RMSE={reg_cv['RMSE']:.3f}  MAE={reg_cv['MAE']:.3f}",
              fontsize=10, fontweight="bold")
ax3.legend(fontsize=8)
ax3.grid(True, alpha=0.2)

# ── Panel 4: Confusion Matrix ─────────────────────────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Non-Anemic", "Anemic"]
)
disp.plot(ax=ax4, colorbar=False, cmap="Blues")
ax4.set_title(
    f"Confusion Matrix\n"
    f"Acc={accuracy:.3f}  F1={f1:.3f}  Sens={recall:.3f}  Spec={specificity:.3f}",
    fontsize=10, fontweight="bold"
)

# ── Panel 5: ROC Curve ────────────────────────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(fpr_arr, tpr_arr, color=BLUE, lw=2,
         label=f"RF model (AUC = {auc_score_val:.3f})")
ax5.plot([0, 1], [0, 1], "k--", lw=1, label="Random (AUC = 0.5)")
ax5.fill_between(fpr_arr, tpr_arr, alpha=0.08, color=BLUE)
ax5.set_xlabel("False Positive Rate (1 − Specificity)", fontsize=10)
ax5.set_ylabel("True Positive Rate (Sensitivity)", fontsize=10)
ax5.set_title("ROC Curve — Anemia Detection", fontsize=10, fontweight="bold")
ax5.legend(fontsize=9)
ax5.set_xlim([0, 1]); ax5.set_ylim([0, 1.02])
ax5.grid(True, alpha=0.2)

# ── Panel 6: Summary bar chart of key metrics ─────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
metric_names  = ["Accuracy", "Precision\n(PPV)", "Sensitivity\n(Recall)",
                  "Specificity", "F1-Score", "AUC-ROC"]
metric_values = [accuracy, ppv, recall, specificity, f1, auc_score_val]
bar_colors = [GREEN if v >= 0.80 else (ORANGE if v >= 0.65 else RED)
              for v in metric_values]
bars = ax6.barh(metric_names, metric_values, color=bar_colors, edgecolor="white", height=0.55)
ax6.axvline(0.80, color="green", lw=1.2, ls="--", alpha=0.7, label="0.80 target")
ax6.axvline(0.70, color=ORANGE,  lw=0.9, ls=":",  alpha=0.7, label="0.70 acceptable")
for bar, val in zip(bars, metric_values):
    ax6.text(val + 0.01, bar.get_y() + bar.get_height()/2,
             f"{val:.3f}", va="center", fontsize=9, fontweight="bold")
ax6.set_xlim([0, 1.12])
ax6.set_xlabel("Score", fontsize=10)
ax6.set_title("Classification Metrics Summary", fontsize=10, fontweight="bold")
ax6.legend(fontsize=8, loc="lower right")
ax6.grid(True, alpha=0.2, axis="x")

plt.savefig(os.path.join(OUTPUT_DIR, "full_evaluation.png"), dpi=150, bbox_inches="tight")
plt.show()
print(f"\n  Saved: {os.path.join(OUTPUT_DIR, 'full_evaluation.png')}")

# =============================================================================
# D) Save results table
# =============================================================================
results_rows = []
# Regression
for name, key, _, _ in rows_reg:
    results_rows.append({"Section": "Regression", "Metric": name.strip(),
                         "CV_Value": reg_cv[key], "Train_Value": reg_train[key]})
# Classification
for name, val, _, note in cls_rows:
    results_rows.append({"Section": "Classification", "Metric": name.strip(),
                         "CV_Value": val, "Train_Value": None, "Notes": note})

eval_df = pd.DataFrame(results_rows)
save_path = os.path.join(OUTPUT_DIR, "evaluation_results.csv")
eval_df.to_csv(save_path, index=False)
print(f"  Saved: {save_path}")
print("\n  Done.")


Extracting /content/Hb_PPG_Dataset.zip ...
Extraction complete.
DATA_DIR : /content/Hb_PPG_Dataset/Hb_PPG_Dataset/data_csv
CSV files: 252

── Wavelength Correction Factors ────────────────────────────────
  w530: device=530nm, dataset=730nm, correction=1.0000
  w660: device=660nm, dataset=660nm, correction=1.0000
  w850: device=850nm, dataset=850nm, correction=1.0000
  w940: device=940nm, dataset=940nm, correction=1.0000
  Development of An Affordable Non-Invasive Anemia Screening Device
  Using Multi-Wavelength Optical Sensing, Edge ML &
  Adaptive Skin-Tone Adaptation

── Hb Label Distribution ────────────────────────────────────────
  Saved: /content/outputs/hb_distribution.png
  Hb range : 8.5 – 17.3 g/dL
  Mean ± SD: 13.9 ± 1.5
  Anemic (<12 g/dL): 14 subjects (5.6%)
Processing 252 subject files...
  [SKIP] 100: excluded
  [SKIP] 132: excluded

  Skipped — no Hb label : 0
  Skipped — poor signal : 0
  Skipped — missing cols: 0

✓ Feature matrix: 250 subjects × 160 columns

── Skin

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


📥 Three files downloaded: hb_rf_model.h, model_meta.json, skin_correction.json
  MODEL EVALUATION — Regression + Anemia Classification

── A) Regression Performance ────────────────────────────────────
  Metric                       5-Fold CV      Training  Notes
  -----------------------------------------------------------------
  RMSE  (g/dL)                     1.153         0.902  lower is better
  MAE   (g/dL)                     0.857         0.656  lower is better
  R²                               0.379         0.620  ≥0.8 excellent, ≥0.6 good
  Pearson r                        0.653         0.845  correlation
  Bias (g/dL)                     +0.017        +0.048  systematic offset
  StdDev diff                      1.153         0.900  random error spread
  LoA lower                       -2.242        -1.716  Bland-Altman −1.96 SD
  LoA upper                       +2.276        +1.813  Bland-Altman +1.96 SD
  Within ±1 g/dL                    61.6          77.2  WHO criteri